# Clase 6 · Predecir la captura con datos del MCP

**IA Aplicada a la Producción Pesquera** — UTN FRCh · PesquerosEnIA

En este notebook nos conectamos **en vivo** al MCP del curso (una fuente de datos pesqueros, sintéticos), traemos los lances recientes y entrenamos un **modelo predictivo**. Es el flujo de Machine Learning de punta a punta: datos → features → modelo → evaluación → interpretación.

> El MCP no tiene autenticación y los datos son sintéticos: podés correrlo sin configurar nada.


## 1. Conectarse al MCP
Definimos un pequeño ayudante para llamar herramientas del MCP. No necesita librerías extra.


In [ ]:
# --- Conexión al MCP del curso (sin dependencias externas) ---
import urllib.request, json
MCP_URL = "https://cursopescafishy-production.up.railway.app/mcp"

def _post(body, sid=None):
    h = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream"}
    if sid: h["Mcp-Session-Id"] = sid
    req = urllib.request.Request(MCP_URL, data=json.dumps(body).encode(), headers=h, method="POST")
    r = urllib.request.urlopen(req, timeout=60)
    data = None
    for line in r.read().decode().splitlines():
        if line.startswith("data: "):
            data = json.loads(line[6:])
    return data, r.headers.get("mcp-session-id")

def mcp_call(tool, args=None):
    """Llama una herramienta del MCP y devuelve una lista de filas (dicts)."""
    _, sid = _post({"jsonrpc":"2.0","id":1,"method":"initialize",
                    "params":{"protocolVersion":"2024-11-05","capabilities":{},
                              "clientInfo":{"name":"colab","version":"1"}}})
    _post({"jsonrpc":"2.0","method":"notifications/initialized"}, sid)
    d, _ = _post({"jsonrpc":"2.0","id":2,"method":"tools/call",
                  "params":{"name":tool,"arguments":args or {}}}, sid)
    res = d.get("result", {})
    sc = res.get("structuredContent")
    if isinstance(sc, dict) and "result" in sc: return sc["result"]
    if isinstance(sc, list): return sc
    rows = []
    for it in res.get("content", []):
        if it.get("type") == "text":
            try: rows.append(json.loads(it["text"]))
            except Exception: pass
    return rows

print("Helper MCP listo ✅")
mcp_call("get_resumen_captura_por_especie")


## 2. Traer los datos
Pedimos los lances de los últimos 30 días con la herramienta `get_capturas_recientes`.


In [ ]:
import pandas as pd

# Traemos los lances de los últimos 30 días directo del MCP
lances = mcp_call("get_capturas_recientes", {"dias": 30})
df = pd.DataFrame(lances)
df["kg_declarados"] = pd.to_numeric(df["kg_declarados"], errors="coerce")
print("Lances traídos del MCP:", df.shape[0])
df[["fecha","embarcacion_nombre","especie","zona_fao","kg_declarados"]].head()


## 3. Explorar
Antes de modelar, miramos cómo se reparte la captura por especie y por zona.


In [ ]:
# ¿Cómo se reparte la captura? (exploración)
print(df.groupby("especie")["kg_declarados"].agg(["count","mean","sum"]).round(0))
print()
print("kg promedio por zona FAO:")
print(df.groupby("zona_fao")["kg_declarados"].mean().round(0))


## 4. Entrenar un modelo predictivo
Predecimos los **kg por lance** a partir de la especie, la embarcación, la zona y la hora. Usamos un *Random Forest* con separación train/test — el flujo estándar de ML.

> Fijate el **R²**: con estas variables el modelo capta parte de la señal, pero no toda.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Feature engineering: especie, embarcación, zona y hora del lance
df["hora_num"] = pd.to_datetime(df["hora"], format="%H:%M:%S", errors="coerce").dt.hour
feat = df.dropna(subset=["kg_declarados"]).copy()
X = pd.get_dummies(feat[["especie","embarcacion_nombre","zona_fao","hora_num"]],
                   columns=["especie","embarcacion_nombre","zona_fao"])
y = feat["kg_declarados"]

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42)
modelo = RandomForestRegressor(n_estimators=300, random_state=42).fit(Xtr, ytr)
pred = modelo.predict(Xte)

print(f"Error medio (MAE): {mean_absolute_error(yte, pred):.0f} kg por lance")
print(f"R² en test:        {r2_score(yte, pred):.2f}")


## 5. ¿Qué variables pesan más?


In [ ]:
import matplotlib.pyplot as plt
imp = pd.Series(modelo.feature_importances_, index=X.columns).sort_values(ascending=False)
imp.head(10).iloc[::-1].plot(kind="barh", color="#0a7fb4")
plt.title("¿Qué variables explican la captura?"); plt.tight_layout(); plt.show()


## 6. Interpretación y conexión con el curso

- **La especie y la zona** suelen ser las variables que más explican la captura — coincide con la intuición del sector.
- Si el **R² quedó modesto**, es la lección clave: *los datos crudos rara vez alcanzan*. El techo del modelo lo define la **capa de datos** (Clase 4): sumar **features ambientales** (SST, clorofila, del simulador del caladero) mejoraría la predicción. Eso es **feature engineering**.
- **Clase 8:** el mismo MCP habilita un **agente** que no solo predice, sino que **actúa** (audita la reconciliación declarado vs. pesado y arma el informe).

*Datos sintéticos vía el MCP "curso pesca fishy" (Damián Giacone).*
